# 05 Regression Modeling — Delivery Time

Compare the promise baseline and three untuned regressors using time-series cross-validation on the training split. Reuse the saved feature sets and the preprocessing definitions from `03_feature_engineering.ipynb`.


## 0. Setup and Load Data

Inputs: `data/fe_data/features_delivery_time.csv` and `data/fe_data/regression_selected_features.json`. The original project module `src/features.py` must be available; preprocessing is imported directly from it.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "XGBoost is required. Install it in this notebook's environment: pip install xgboost"
    ) from exc

RANDOM_STATE = 42
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "fe_data" / "features_delivery_time.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebook directory; feature CSV not found.")
if not (ROOT / "src" / "features.py").is_file():
    raise FileNotFoundError(
        "Missing src/features.py. Restore the original helper used by "
        "03_feature_engineering.ipynb; do not recreate its preprocessing rules."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import build_preprocessor

DATA_DIR = ROOT / "data" / "fe_data"
df = pd.read_csv(DATA_DIR / "features_delivery_time.csv")
selected_features = json.loads(
    (DATA_DIR / "regression_selected_features.json").read_text(encoding="utf-8")
)
print("Dataset shape:", df.shape)
print("Feature configuration loaded successfully.")


Dataset shape: (96454, 72)
Feature configuration loaded successfully.


## 1. Define Prediction Target

All four models predict `log(delivery_days / promised_delivery_days)`. This measures relative delivery performance against the platform's promised delivery time, because raw `delivery_days` shows temporal drift. Convert predictions back to delivery days before calculating any evaluation metric.


In [2]:
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

assert df["delivery_days"].notna().all()
assert df["promised_delivery_days"].notna().all()
assert (df["delivery_days"] > 0).all()
assert (df["promised_delivery_days"] > 0).all()

df["target_log_ratio"] = np.log(
    df["delivery_days"] / df["promised_delivery_days"]
)

def to_delivery_days(pred_log_ratio, promised_days):
    return promised_days * np.exp(pred_log_ratio)


## 2. Prepare Time-Ordered Train / Validation / Test

Preserve the existing split labels and chronological order. Validation and test are reserved for the later sections.


In [3]:
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

assert not train_df.empty and not val_df.empty and not test_df.empty
assert train_df["order_purchase_timestamp"].max() <= val_df["order_purchase_timestamp"].min()
assert val_df["order_purchase_timestamp"].max() <= test_df["order_purchase_timestamp"].min()

split_summary = pd.DataFrame([
    {
        "split": name,
        "n": len(frame),
        "start_date": frame["order_purchase_timestamp"].min().date(),
        "end_date": frame["order_purchase_timestamp"].max().date(),
    }
    for name, frame in [
        ("train", train_df), ("validation", val_df), ("test", test_df)
    ]
])
display(split_summary)


,split,n,start_date,end_date
0,train,83947,2016-09-15,2018-06-30
1,validation,6156,2018-07-01,2018-07-31
2,test,6351,2018-08-01,2018-08-29


## 3. Load Selected Feature Sets

Use both configurations exactly as saved in the JSON. Deduplicate only the original input column lists; keep each transformation group intact.


In [4]:
def original_columns(configuration):
    return list(dict.fromkeys(
        column for columns in configuration.values() for column in columns
    ))

linear_features = original_columns(selected_features["linear"])
tree_features = original_columns(selected_features["tree"])

print("Linear selected features:", len(linear_features))
print("Tree selected features:", len(tree_features))


Linear selected features: 33
Tree selected features: 37


## 4. Build Preprocessing Pipelines

Reuse `build_preprocessor` from 03 without changing its definitions. Preprocessors are fitted only inside model pipelines on each fold's training rows. Fixed band definitions stay unchanged; rare-category mappings are learned from fold training only.


### 4.1 Linear Preprocessing

The existing helper applies median imputation and `StandardScaler` to regular numeric columns; median imputation, `log1p` and scaling to skewed columns; and fixed bands followed by one-hot encoding to banded columns. Categorical preprocessing groups rare categories with fewer than 20 training observations and uses one-hot encoding that safely handles unknown categories.

`estimated_mean_distance_km` intentionally appears as both a skewed continuous feature and a banded feature. Both representations are retained.


In [5]:
linear_preprocessor = build_preprocessor(
    **selected_features["linear"]
)


### 4.2 Tree Preprocessing

Tree splits learn nonlinear thresholds directly, so the saved tree configuration needs no scaling, `log1p` or artificial banding. Reuse the helper with `scale=False`. Random Forest and XGBoost share the same preprocessing definition, with independent copies fitted inside their pipelines.


In [6]:
tree_preprocessor = build_preprocessor(
    **selected_features["tree"],
    scale=False,
)


## 5. Define Promise Baseline

As in 03, the baseline predicts the median training log ratio. In delivery days this is `promised_delivery_days × exp(median training log ratio)`. Re-estimate the median separately within every CV fold.


In [7]:
class PromiseBaseline(RegressorMixin, BaseEstimator):
    """Predict a constant log ratio estimated from the fold's training target."""

    def fit(self, X, y):
        self.median_ = float(np.median(y))
        return self

    def predict(self, X):
        return np.full(len(X), self.median_)

promise_baseline = PromiseBaseline()


## 6. Define Untuned Candidate Models

Use default model hyperparameters, setting only the random seed, parallelism and XGBoost regression objective where relevant. All models learn the same log-ratio target.


### 6.1 Linear Regression


In [8]:
linear_model = Pipeline([
    ("preprocessor", clone(linear_preprocessor)),
    ("model", LinearRegression()),
])


### 6.2 Random Forest


In [9]:
random_forest_model = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])


### 6.3 XGBoost


In [10]:
xgboost_model = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])


## 7. Untuned Model Comparison

Evaluate only `train_df`. MAE, RMSE and bias are in delivery days; R² is also computed against actual delivery days. Bias is `mean(predicted_days - actual_days)`: positive values indicate overprediction and negative values indicate underprediction.


### 7.1 Time-Series Cross-Validation

Materialize five expanding-window folds once and reuse those exact boundaries for all four models. Each fold clones and fits its model, including preprocessing or the baseline median, using only the earlier training portion.


In [11]:
tscv = TimeSeriesSplit(n_splits=5)
cv_folds = list(tscv.split(train_df))

candidate_models = {
    "Promise Baseline": (promise_baseline, ["promised_delivery_days"]),
    "Linear Regression": (linear_model, linear_features),
    "Random Forest": (random_forest_model, tree_features),
    "XGBoost": (xgboost_model, tree_features),
}

def evaluate_time_series_cv(models, frame, folds):
    """Return fold metrics in delivery days using shared chronological splits."""
    records = []
    for name, (model, columns) in models.items():
        for fold, (fit_indices, eval_indices) in enumerate(folds, start=1):
            fold_train = frame.iloc[fit_indices]
            fold_eval = frame.iloc[eval_indices]
            fitted_model = clone(model).fit(
                fold_train[columns], fold_train["target_log_ratio"]
            )
            pred_log_ratio = fitted_model.predict(fold_eval[columns])
            pred_days = to_delivery_days(
                pred_log_ratio, fold_eval["promised_delivery_days"].to_numpy()
            )
            actual_days = fold_eval["delivery_days"].to_numpy()
            records.append({
                "model": name,
                "fold": fold,
                "mae": mean_absolute_error(actual_days, pred_days),
                "rmse": np.sqrt(mean_squared_error(actual_days, pred_days)),
                "r2": r2_score(actual_days, pred_days),
                "bias": np.mean(pred_days - actual_days),
            })
    return pd.DataFrame(records)

cv_results = evaluate_time_series_cv(candidate_models, train_df, cv_folds)


### 7.2 Compare Baseline vs Untuned Models

CV mean MAE is the primary untuned comparison metric. CV standard deviation indicates temporal stability. Count a win only when the model's MAE is strictly lower than the baseline on the same fold. Validation and test remain untouched at this stage.


In [12]:
cv_summary = cv_results.groupby("model").agg(
    mae_mean=("mae", "mean"),
    mae_sd=("mae", "std"),
    rmse_mean=("rmse", "mean"),
    r2_mean=("r2", "mean"),
    bias_mean=("bias", "mean"),
)

fold_mae = cv_results.pivot(index="fold", columns="model", values="mae")
fold_wins = fold_mae.lt(fold_mae["Promise Baseline"], axis=0).sum()
cv_summary["folds_beating_baseline"] = fold_wins.reindex(cv_summary.index)
cv_summary.loc["Promise Baseline", "folds_beating_baseline"] = np.nan

baseline_mean_mae = cv_summary.loc["Promise Baseline", "mae_mean"]
cv_summary["mae_improvement_pct"] = (
    (baseline_mean_mae - cv_summary["mae_mean"]) / baseline_mean_mae * 100
)

cv_summary = (
    cv_summary.sort_values("mae_mean")
    .rename(columns={
        "mae_mean": "CV MAE Mean",
        "mae_sd": "CV MAE SD",
        "rmse_mean": "CV RMSE Mean",
        "r2_mean": "CV R2 Mean",
        "bias_mean": "CV Bias Mean",
        "folds_beating_baseline": "Folds Beating Baseline",
        "mae_improvement_pct": "MAE improvement vs baseline (%)",
    })
    .rename_axis("Model")
    .reset_index()
)
display(cv_summary.round(3))


,Model,CV MAE Mean,CV MAE SD,CV RMSE Mean,CV R2 Mean,CV Bias Mean,Folds Beating Baseline,MAE improvement vs baseline (%)
0,Random Forest,5.169,0.953,8.312,0.217,-1.587,5.0,10.170
1,XGBoost,5.183,0.937,8.352,0.209,-1.632,5.0,9.927
2,Linear Regression,5.210,0.864,8.523,0.171,-2.363,4.0,9.458
3,Promise Baseline,5.755,1.065,9.167,0.049,-2.448,NaN,0.000


## 8. Hyperparameter Tuning


### 8.1 Random Forest


### 8.2 XGBoost


## 9. Tuned Model Comparison


### 9.1 Time-Series Cross-Validation


### 9.2 Compare Untuned vs Tuned Models


## 10. Validation Evaluation


## 11. Final Model Selection


## 12. Refit on Train + Validation


## 13. Final Test Evaluation


## 14. Error Analysis


## 15. Model Interpretation and Feature Importance


## 16. Save Results
